In [1]:
# pip install torchmetrics
%pip install -q dagshub mlflow torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.0/261.0 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.7/24.7 MB 73.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 1.9 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 30.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.2 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [6]:
import dagshub
dagshub.init(repo_owner='pratham.doshi', repo_name='faster_rcnn_with_mlflow', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=14d5212c-e616-408c-8f12-3a603112b21c&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=beb17668775b0a840765b9bcc03c190d45fd86dc639752d91ce8c888a804118d




Accessing as pratham.doshi

Initialized MLflow to track repo "pratham.doshi/faster_rcnn_with_mlflow"

Repository pratham.doshi/faster_rcnn_with_mlflow initialized!

In [2]:
import mlflow
mlflow.set_experiment("trial_experiment")

In [10]:
import os
import xml.etree.ElementTree as ET
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

import torch
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

from torchvision import transforms as T
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.ops import box_convert

from torchmetrics.detection.mean_ap import MeanAveragePrecision

from tqdm import tqdm
import mlflow


In [11]:
class YoloDataset(Dataset):
    def __init__(self, img_dir, label_dir, transforms=None):
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.transforms = transforms
        self.images = sorted([f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.jpeg', '.png'))])
        print(f"Found {len(self.images)} images")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert("RGB")
        width, height = image.size

        label_file = img_name.rsplit('.', 1)[0] + '.txt'
        label_path = os.path.join(self.label_dir, label_file)

        boxes = []
        labels = []

        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f:
                    if line.strip() == "":
                        continue
                    class_id, x_center, y_center, w, h = map(float, line.strip().split())

                    # Convert YOLO normalized format to absolute [xmin, ymin, xmax, ymax]
                    x_center *= width
                    y_center *= height
                    w *= width
                    h *= height

                    xmin = x_center - w / 2
                    ymin = y_center - h / 2
                    xmax = x_center + w / 2
                    ymax = y_center + h / 2

                    boxes.append([xmin, ymin, xmax, ymax])
                    labels.append(int(class_id) + 1)  # Faster R-CNN expects class labels starting from 1

        if len(boxes) == 0:
            return None  # You can also handle skipping None in DataLoader with custom collate_fn

        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64)

        target = {
            'boxes': boxes,
            'labels': labels,
            'image_id': torch.tensor([idx]),  # required for evaluation
            'area': (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0]),
            'iscrowd': torch.zeros((len(boxes),), dtype=torch.int64),  # assuming all instances are not crowd
        }

        if self.transforms:
            image = self.transforms(image)

        return image, target

transform = T.ToTensor()

In [12]:
def collate_fn(batch):
    batch = list(filter(lambda x: x is not None, batch))  # remove None entries
    return tuple(zip(*batch))

train_dataset = YoloDataset(
    # img_dir='/content/drive/MyDrive/PPE Detection.v1i.yolov8/train/images',
    # /kaggle/input/ppe-data/train/images
    img_dir="/kaggle/input/ppe-data/train/images",
    label_dir='/kaggle/input/ppe-data/train/labels',
    transforms=transform
)

valid_dataset = YoloDataset(
    img_dir='/kaggle/input/ppe-data/valid/images',
    label_dir='/kaggle/input/ppe-data/valid/labels',
    transforms=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=50,
    shuffle=True,
    num_workers=2,
    collate_fn=collate_fn
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=50,
    shuffle=False,
    num_workers=2,
    collate_fn=collate_fn
)

Found 7133 images
Found 311 images


In [13]:
test_dataset = YoloDataset(
    img_dir='/kaggle/input/ppe-data/test/images',
    label_dir='/kaggle/input/ppe-data/test/labels',
    transforms=transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    collate_fn=collate_fn
)


Found 139 images


For training from scratch

In [14]:
# model = fasterrcnn_resnet50_fpn(pretrained=True)
# num_classes = 5  # background + class
# in_features = model.roi_heads.box_predictor.cls_score.in_features
# model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Using already trained model from model log in mlflow

In [16]:
num_classes = 5
model = fasterrcnn_resnet50_fpn(weights=None)

in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

path = mlflow.artifacts.download_artifacts(artifact_uri =  "mlflow-artifacts:/89868de5557344b4b4b398881f66835a/b35e32fff6ad4614a3b3436f72696fdf/artifacts/best_model.pth")
checkpoint = torch.load(path, map_location="cpu")

model.load_state_dict(checkpoint['model_state_dict'])

params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.SGD(params, lr=0.001, momentum=0.9, weight_decay=0.0005)

optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
lr_scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

epoch = checkpoint['epoch']
loss = checkpoint['loss']

print(f"✅ Model loaded from epoch {epoch} with loss {loss:.4f}")
model.eval()


KeyError: 'epoch'

In [18]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
model = model.to(device)

if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Memory Allocated: {torch.cuda.memory_allocated(0)/(1024**2):.2f} MB")

Using device: cuda:0
GPU Name: Tesla P100-PCIE-16GB
Memory Allocated: 161.77 MB


In [19]:
import torch
from tqdm import tqdm
from torch.nn.utils import clip_grad_norm_

def train_one_epoch(model, optimizer, data_loader, device):
    model.train()
    total_loss = 0.0
    num_batches = 0

    progress_bar = tqdm(data_loader, desc="Training", leave=True)

    for batch_idx, (images, targets) in enumerate(progress_bar):
        try:
           
            if not images or not targets:
                print(f" Skipping batch {batch_idx}: Empty images or targets")
                continue

            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            if any(t['boxes'].numel() == 0 for t in targets):
                print(f" Skipping batch {batch_idx}: No bounding boxes in targets")
                continue

            if num_batches == 0:
                print("\nFirst batch debug info:")
                print(f"   ➤ Number of images: {len(images)}")
                print(f"   ➤ Image shape: {images[0].shape}")
                print(f"   ➤ Target boxes shape: {targets[0]['boxes'].shape}\n")

            optimizer.zero_grad()

            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())

            if not torch.isfinite(losses):
                print(f" Invalid loss at batch {batch_idx}: {losses.item():.4f}")
                continue

            losses.backward()
            clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            # 📊Track and display loss
            total_loss += losses.item()
            num_batches += 1

            progress_bar.set_postfix({
                "Batch Loss": f"{losses.item():.4f}",
                "Avg Loss": f"{total_loss / num_batches:.4f}"
            })

        except Exception as e:
            print(f" Exception in batch {batch_idx}: {repr(e)}")
            continue

    progress_bar.close()

    if num_batches == 0:
        print("No valid batches were processed in this epoch.")
        return float("inf")

    avg_loss = total_loss / num_batches
    print(f"\nEpoch complete. Avg Loss: {avg_loss:.4f}\n")
    return avg_loss


In [20]:
def run_training(model, optimizer, lr_scheduler, train_loader, device, num_epochs):
    best_loss = float("inf")

    with mlflow.start_run(run_name="Training", nested=True):
        mlflow.log_params({
            "model": "fasterrcnn_resnet50_fpn",
            "num_classes": num_classes,
            "learning_rate": param_group["initial_lr"],
            "momentum": param_group["momentum"],
            "weight_decay": param_group["weight_decay"],
            "lr_scheduler_step": lr_scheduler.step_size,
            "lr_scheduler_gamma":lr_scheduler.gamma,
            "epochs": num_epochs,
            "batch_size": train_loader.batch_size,
        })

        for epoch in range(num_epochs):
            print(f"\n Epoch {epoch+1}/{num_epochs}")
            epoch_loss = train_one_epoch(model, optimizer, train_loader, device)
            lr_scheduler.step()
            current_lr = lr_scheduler.get_last_lr()[0]

            mlflow.log_metric("train_loss", epoch_loss, step=epoch)
            mlflow.log_metric("lr", current_lr, step=epoch)

            if epoch_loss < best_loss:
                best_loss = epoch_loss
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'loss': epoch_loss,
                }, "best_model.pth")
                mlflow.log_artifact("best_model.pth")
                print(" Best model saved and logged!")

        mlflow.log_metric("best_train_loss", best_loss)

    mlflow.pytorch.log_model(model, name="model")
    return best_loss


This will evalute and save model prediction on test data

In [21]:
def run_evaluation(model, test_loader, class_names, device, score_threshold=0.5, image_dir=None, output_dir="predictions"):
    

    model.eval()
    metric = MeanAveragePrecision()
    os.makedirs(output_dir, exist_ok=True)

    image_files = sorted([
        f for f in os.listdir(image_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ])

    with mlflow.start_run(run_name="Evaluation", nested=True):
        mlflow.log_param("score_threshold", score_threshold)
        mlflow.log_param("num_classes", len(class_names))
        mlflow.log_param("test_batch_size", test_loader.batch_size)

        with torch.no_grad():
            for batch_idx, (images, targets) in enumerate(tqdm(test_loader, desc="Evaluating")):
                images = [img.to(device) for img in images]
                targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
                outputs = model(images)

                preds = []
                gts = []
                for pred, gt in zip(outputs, targets):
                    preds.append({
                        "boxes": pred["boxes"].cpu(),
                        "scores": pred["scores"].cpu(),
                        "labels": pred["labels"].cpu(),
                    })
                    gts.append({
                        "boxes": gt["boxes"].cpu(),
                        "labels": gt["labels"].cpu(),
                    })
                metric.update(preds, gts)

                for i, (image_tensor, pred) in enumerate(zip(images, outputs)):
                    image_name = image_files[batch_idx * test_loader.batch_size + i]
                    image_path = os.path.join(image_dir, image_name)
                    image_pil = Image.open(image_path).convert("RGB")

                    fig, ax = plt.subplots(figsize=(12, 8))
                    ax.imshow(image_pil)

                    for box, label, score in zip(pred["boxes"], pred["labels"], pred["scores"]):
                        if score < score_threshold:
                            continue
                        xmin, ymin, xmax, ymax = box.tolist()
                        rect = patches.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin,
                                                 linewidth=2, edgecolor='lime', facecolor='none')
                        ax.add_patch(rect)
                        label_text = class_names[label.item() - 1]
                        ax.text(xmin, ymin - 5, f"{label_text}: {score:.2f}",
                                fontsize=12, color='white',
                                bbox=dict(facecolor='green', alpha=0.5))

                    ax.axis('off')
                    plt.tight_layout()
                    save_path = os.path.join(output_dir, f"pred_{image_name}")
                    plt.savefig(save_path)
                    plt.close()

                    if i == 0:
                        mlflow.log_artifact(save_path)

        results = metric.compute()
        print("\nEvaluation Summary:")
        for k, v in results.items():
            if isinstance(v, torch.Tensor) and v.ndim == 0:
                mlflow.log_metric(k, v.item())
                print(f"{k}: {v.item():.4f}")


In [22]:
class_names = ['gloves', 'googles', 'helmet', 'jacket']

In [23]:
with mlflow.start_run(run_name="faster_rcnn") as parent_run:
    best_loss = run_training(model, optimizer, lr_scheduler, train_loader, device, num_epochs=10)
    run_evaluation(model, test_loader, class_names, device, score_threshold=0.5,
                   image_dir="/kaggle/input/ppe-data/test/images",
                   output_dir="/kaggle/working/predictions")


Evaluating: 100%|██████████| 35/35 [00:47<00:00,  1.35s/it]



📊 Evaluation Summary:
map: 0.5185
map_50: 0.8994
map_75: 0.5289
map_small: 0.2152
map_medium: 0.4492
map_large: 0.6434
mar_1: 0.5401
mar_10: 0.5770
mar_100: 0.5770
mar_small: 0.2857
mar_medium: 0.5119
mar_large: 0.7107
map_per_class: -1.0000
mar_100_per_class: -1.0000
🏃 View run Evaluation at: https://dagshub.com/pratham.doshi/faster_rcnn_with_mlflow.mlflow/#/experiments/0/runs/5a6571c6985e4d92af6ef3126ba8d1d8
🧪 View experiment at: https://dagshub.com/pratham.doshi/faster_rcnn_with_mlflow.mlflow/#/experiments/0
🏃 View run faster_rcnn at: https://dagshub.com/pratham.doshi/faster_rcnn_with_mlflow.mlflow/#/experiments/0/runs/266f5ec5075848228eda6e356a4c7445
🧪 View experiment at: https://dagshub.com/pratham.doshi/faster_rcnn_with_mlflow.mlflow/#/experiments/0
